# Imports and data paths

In [ ]:
from pathlib import Path
import gc
import json
import random
import shutil
import time

import networkx as nx
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from scipy.spatial.distance import cdist
from torch import nn
from torch_geometric.nn import GCNConv, GraphConv, SAGEConv

# find file
def find_file(name):
    matches = sorted(Path("/kaggle/input").rglob(name))
    if not matches:
        raise FileNotFoundError(f"Could not find {name} in /kaggle/input")
    return matches[0]

ID = "SEZ21_ID"
DATES = ["2022-06-17", "2023-08-15", "2024-08-25"]
SEEDS = [42, 123, 2026, 3407, 777]
FEATURE_GROUPS = {
    "all": ["ndvi", "ndbi", "ndwi", "albedo", "svf", "bcr", "tcd", "pop"],
    "public": ["ndvi", "ndbi", "ndwi", "albedo", "bcr", "tcd"],
}
K = 8
COVERAGE = 0.8

MASTER = find_file("node_features_all_dates_clean_v2.csv")
CENTROIDS = find_file("zone_centroids.csv")
SNAPSHOT = Path("/kaggle/input/dataset-gnn/results_static_3000_v1_backup/results_static_3000_v1/all/data_snapshot")
if not SNAPSHOT.exists():
    SNAPSHOT = find_file("static_node_table.csv").parent

DATA_OUT = Path("/kaggle/working/static_three_dates_v1")
DATA_OUT.mkdir(parents=True, exist_ok=True)

print("Master table:", MASTER)
print("Snapshot dir:", SNAPSHOT)

## Align three dates, restrict splits, and build both graphs

In [ ]:
# Load original 2022 node order and master feature table
nodes = pd.read_csv(SNAPSHOT / "static_node_table.csv", dtype={ID: str})
nodes[ID] = nodes[ID].str.strip()
old_ids = nodes[ID].tolist()
n_old = len(nodes)

df = pd.read_csv(MASTER, dtype={ID: str})
df[ID] = df[ID].str.strip()
date_text = df["date"].astype(str).str.strip().str.replace(r"^(\d{4})(\d{2})(\d{2})$", r"\1-\2-\3", regex=True)
df["date"] = pd.to_datetime(date_text).dt.strftime("%Y-%m-%d")

variables = ["lst"] + FEATURE_GROUPS["all"]
for col in variables + [f"{v}_coverage" for v in variables]:
    df[col] = pd.to_numeric(df[col])

# Filter nodes that are valid on all three dates
daily = {}
valid_masks = []
for date in DATES:
    day = df[df["date"] == date].set_index(ID).loc[old_ids].copy()
    vals = day[variables].to_numpy(dtype=float)
    cov = day[[f"{v}_coverage" for v in variables]].to_numpy(dtype=float)
    valid = np.isfinite(vals).all(axis=1) & (cov >= COVERAGE).all(axis=1)
    daily[date] = day
    valid_masks.append(valid)

keep = np.logical_and.reduce(valid_masks)
kept_old_indices = np.flatnonzero(keep)
n = int(keep.sum())
print(f"Common valid nodes across all 3 dates: {n} / {n_old}")

old_to_new = np.full(n_old, -1, dtype=np.int64)
old_to_new[keep] = np.arange(n)

mapping = pd.DataFrame({
    "local_node_id": np.arange(n),
    "previous_local_node_id": kept_old_indices,
    ID: nodes.loc[keep, ID].to_numpy(),
})
if "original_node_id" in nodes.columns:
    mapping["original_node_id"] = nodes.loc[keep, "original_node_id"].to_numpy()
new_ids = mapping[ID].tolist()

# Restrict original split masks to common nodes
masks_by_seed = {}
for seed in SEEDS:
    with np.load(SNAPSHOT / f"split_{seed}.npz") as z:
        masks = {name: np.asarray(z[name]).astype(bool)[keep] for name in ["mask_train", "mask_valid", "mask_test"]}
    assert (masks["mask_train"].astype(int) + masks["mask_valid"].astype(int) + masks["mask_test"].astype(int) == 1).all()
    masks_by_seed[seed] = masks

def to_bidirectional(pairs):
    return np.concatenate([pairs.T, pairs[:, ::-1].T], axis=1).astype(np.int64)

# 1) Induce shared-boundary graph on common nodes
old_edge = np.load(SNAPSHOT / "edge_index_contiguity_static.npy")
edge_keep = keep[old_edge[0]] & keep[old_edge[1]]
remapped = old_to_new[old_edge[:, edge_keep]]
pairs = np.sort(remapped.T, axis=1)
pairs = np.unique(pairs[pairs[:, 0] != pairs[:, 1]], axis=0)
boundary_edge = to_bidirectional(pairs)

# 2) Build Euclidean kNN graph with Gaussian weights
centroids = pd.read_csv(CENTROIDS, dtype={ID: str})
centroids[ID] = centroids[ID].str.strip()
centroids = centroids.set_index(ID).loc[new_ids]
xy = centroids[["x", "y"]].to_numpy(dtype=float)

dist_mat = cdist(xy, xy, metric="euclidean")
np.fill_diagonal(dist_mat, np.inf)
knn_idx = np.argsort(dist_mat, axis=1, kind="stable")[:, :K]

dir_pairs = np.column_stack([np.repeat(np.arange(n), K), knn_idx.reshape(-1)])
euclidean_pairs, sel_counts = np.unique(np.sort(dir_pairs, axis=1), axis=0, return_counts=True)

edge_distances = dist_mat[euclidean_pairs[:, 0], euclidean_pairs[:, 1]]
sigma = float(np.median(edge_distances[edge_distances > 0]))
weights32 = np.exp(-(edge_distances / sigma) ** 2).astype(np.float32)

euclidean_edge = to_bidirectional(euclidean_pairs)
euclidean_weight = np.concatenate([weights32, weights32])

# Summarize both graphs
def describe_graph(name, undirected_pairs):
    G = nx.Graph()
    G.add_nodes_from(range(n))
    G.add_edges_from(undirected_pairs.tolist())
    deg = np.array([G.degree(i) for i in range(n)])
    comps = list(nx.connected_components(G))
    return {
        "graph": name, "nodes": n, "undirected_edges": G.number_of_edges(),
        "mean_degree": float(deg.mean()), "min_degree": int(deg.min()),
        "max_degree": int(deg.max()), "isolated_nodes": int((deg == 0).sum()),
        "components": len(comps), "largest_component": max(map(len, comps)),
    }

graph_stats = pd.DataFrame([
    describe_graph("shared_boundary", pairs),
    describe_graph("euclidean_k8_gaussian", euclidean_pairs),
])
display(graph_stats)

## Save prepared graphs, masks, and standardized splits

In [ ]:
# Save node order, excluded nodes, graphs, and masks
mapping["x"] = xy[:, 0]
mapping["y"] = xy[:, 1]
mapping["crs_epsg"] = 32632
mapping.to_csv(DATA_OUT / "node_order.csv", index=False)
nodes.loc[~keep, ["local_node_id", ID]].to_csv(DATA_OUT / "excluded_nodes.csv", index=False)

graphs_dir = DATA_OUT / "graphs"
graphs_dir.mkdir(exist_ok=True)
np.save(graphs_dir / "edge_index_contiguity_static.npy", boundary_edge)
np.save(graphs_dir / "edge_index_euclidean_k8.npy", euclidean_edge)
np.save(graphs_dir / "edge_weight_euclidean_k8.npy", euclidean_weight)
graph_stats.to_csv(graphs_dir / "graph_summary.csv", index=False)

pd.DataFrame({
    "source": euclidean_pairs[:, 0], "target": euclidean_pairs[:, 1],
    "distance_m": edge_distances, "weight": weights32,
    "mutual_selection": sel_counts == 2,
}).to_csv(graphs_dir / "euclidean_edges.csv", index=False)

masks_dir = DATA_OUT / "masks"
masks_dir.mkdir(exist_ok=True)
for seed, masks in masks_by_seed.items():
    np.savez_compressed(masks_dir / f"split_{seed}.npz", **masks)

# Save data config
prep_config = {
    "experiment": "independent_static_experiments_on_common_nodes",
    "dates": DATES, "nodes": n, "feature_groups": FEATURE_GROUPS,
    "split_seeds": SEEDS, "coverage_threshold": COVERAGE,
    "standardization": "per_date_per_split_training_nodes_only",
    "euclidean_graph": {"crs_epsg": 32632, "k": K, "symmetrization": "union", "sigma_m": sigma},
}
with open(DATA_OUT / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(prep_config, f, indent=2)

# Standardize X and y using training nodes only for each date, group, and split
for date in DATES:
    day = daily[date].loc[new_ids].copy()
    y_raw = day["lst"].to_numpy(dtype=np.float64)

    node_table = day.reset_index()
    node_table.insert(0, "local_node_id", np.arange(n))
    node_table["previous_local_node_id"] = kept_old_indices
    node_table["date"] = date
    node_table["label_valid"] = True

    for group, feature_names in FEATURE_GROUPS.items():
        folder = DATA_OUT / date / group
        folder.mkdir(parents=True, exist_ok=True)
        node_table.to_csv(folder / "static_node_table.csv", index=False)
        with open(folder / "feature_names.json", "w", encoding="utf-8") as f:
            json.dump(feature_names, f, indent=2)

        X_raw = day[feature_names].to_numpy(dtype=np.float64)

        for seed, masks in masks_by_seed.items():
            train = masks["mask_train"]
            x_mean = X_raw[train].mean(axis=0)
            x_std = X_raw[train].std(axis=0, ddof=0)
            x_std = np.where(x_std < 1e-8, 1.0, x_std)

            y_mean = float(y_raw[train].mean())
            y_std = float(y_raw[train].std(ddof=0))

            X = ((X_raw - x_mean) / x_std).astype(np.float32)
            y = ((y_raw - y_mean) / y_std).astype(np.float32)

            np.savez_compressed(
                folder / f"split_{seed}.npz",
                X=X, y=y, X_raw=X_raw, y_raw=y_raw,
                x_mean=x_mean, x_std=x_std,
                y_mean=np.array(y_mean), y_std=np.array(y_std),
                feature_names=np.array(feature_names, dtype=str),
                section_ids=np.array(new_ids, dtype=str),
                **masks
            )

shutil.make_archive(str(DATA_OUT) + "_backup", "zip", root_dir=DATA_OUT.parent, base_dir=DATA_OUT.name)
print("Prepared dataset saved and zipped.")

## Training configuration, data loaders, and model definition

In [ ]:
DATA = Path("/kaggle/working/static_three_dates_v1")
OUT = Path("/kaggle/working/results_static_three_dates_v1")
OUT.mkdir(parents=True, exist_ok=True)

RUN_DATES = ["2022-06-17", "2023-08-15", "2024-08-25"]
ALL_DATES = ["2022-06-17", "2023-08-15", "2024-08-25"]
GROUPS = ["all", "public"]
SPLIT_SEEDS = [42, 123, 2026, 3407, 777]
FINAL_SEEDS = [11, 22, 33]

N_TRIALS = 20
TUNING_SEED = 11
TUNE_MAX_EPOCHS = 1500
FINAL_MAX_EPOCHS = 3000
DROPOUT = 0.1
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0

EXPERIMENTS = {
    "MLP": {"model": "MLP", "graph": "none"},
    "GraphSAGE_contiguity": {"model": "GraphSAGE", "graph": "contiguity"},
    "GCN_contiguity": {"model": "GCN", "graph": "contiguity"},
    "GraphConv_contiguity": {"model": "GraphConv", "graph": "contiguity"},
    "GCN_euclidean": {"model": "GCN", "graph": "euclidean"},
    "GraphConv_euclidean": {"model": "GraphConv", "graph": "euclidean"},
}
MODEL_INDEX = {"MLP": 0, "GraphSAGE": 1, "GCN": 2, "GraphConv": 3}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Using device:", DEVICE)
###
# Copy dataset snapshot into results folder
SOURCE = OUT / "data_snapshot"
shutil.copytree(DATA, SOURCE, dirs_exist_ok=True)
node_order = pd.read_csv(SOURCE / "node_order.csv", dtype={"SEZ21_ID": str})
N = len(node_order)

# Load graphs to device
boundary_np = np.load(SOURCE / "graphs/edge_index_contiguity_static.npy").astype(np.int64)
euclidean_np = np.load(SOURCE / "graphs/edge_index_euclidean_k8.npy").astype(np.int64)
weight_np = np.load(SOURCE / "graphs/edge_weight_euclidean_k8.npy").astype(np.float32)

GRAPHS = {
    "none": (None, None),
    "contiguity": (torch.tensor(boundary_np, dtype=torch.long, device=DEVICE), None),
    "euclidean": (
        torch.tensor(euclidean_np, dtype=torch.long, device=DEVICE),
        torch.tensor(weight_np, dtype=torch.float32, device=DEVICE),
    ),
}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def load_data(date, group, split):
    folder = SOURCE / date / group
    features = read_json(folder / "feature_names.json")
    with np.load(folder / f"split_{split}.npz") as z:
        X = z["X"].astype(np.float32)
        y = z["y"].reshape(-1).astype(np.float32)
        y_raw = z["y_raw"].reshape(-1).astype(np.float64)
        y_mean = float(z["y_mean"].item())
        y_std = float(z["y_std"].item())
        masks = {
            "train": z["mask_train"].astype(bool),
            "val": z["mask_valid"].astype(bool),
            "test": z["mask_test"].astype(bool),
        }
    return {
        "X": torch.tensor(X, device=DEVICE),
        "y": torch.tensor(y, device=DEVICE),
        "y_raw": y_raw, "y_mean": y_mean, "y_std": y_std,
        "features": features, "masks_np": masks,
        "masks": {k: torch.tensor(m, device=DEVICE) for k, m in masks.items()},
    }
################################################################################
class RegressionModel(nn.Module):
    def __init__(self, model_name, input_dim, params):
        super().__init__()
        self.model_name = model_name
        self.activation = params["activation"]
        self.layers = nn.ModuleList()

        hidden_dim = int(params["hidden_dim"])
        n_layers = int(params["n_layers"])

        for i in range(n_layers):
            in_dim = input_dim if i == 0 else hidden_dim
            if model_name == "MLP":
                layer = nn.Linear(in_dim, hidden_dim)
            elif model_name == "GraphSAGE":
                layer = SAGEConv(in_dim, hidden_dim, aggr="mean")
            elif model_name == "GCN":
                layer = GCNConv(in_dim, hidden_dim, cached=True, add_self_loops=True, normalize=True)
            elif model_name == "GraphConv":
                layer = GraphConv(in_dim, hidden_dim, aggr="add")
            else:
                raise ValueError(f"Unknown model: {model_name}")
            self.layers.append(layer)

        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index=None, edge_weight=None):
        for layer in self.layers:
            if self.model_name == "MLP":
                x = layer(x)
            elif self.model_name == "GraphSAGE":
                x = layer(x, edge_index)
            else:
                x = layer(x, edge_index, edge_weight=edge_weight)

            if self.activation == "relu":
                x = F.relu(x)
            elif self.activation == "elu":
                x = F.elu(x)
            else:
                x = F.gelu(x)

            x = F.dropout(x, p=DROPOUT, training=self.training)
        return self.output(x).squeeze(-1)

## Training, tuning, and metric functions

In [ ]:
def regression_metrics(observed, predicted):
    err = predicted - observed
    mse = float(np.mean(err ** 2))
    sst = float(np.sum((observed - observed.mean()) ** 2))
    return {
        "n": int(len(observed)),
        "MAE": float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(mse)),
        "R2": float(1 - np.sum(err ** 2) / sst) if sst > 0 else np.nan,
        "Bias": float(err.mean()),
    }

#############################################################
def fit_one(d, experiment, params, init_seed, max_epochs, folder):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    seed_everything(init_seed)

    spec = EXPERIMENTS[experiment]
    edge_index, edge_weight = GRAPHS[spec["graph"]]
    model = RegressionModel(spec["model"], d["X"].shape[1], params).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(params["learning_rate"]), weight_decay=WEIGHT_DECAY)

    train_mask, val_mask = d["masks"]["train"], d["masks"]["val"]
    best_val = float("inf")
    best_epoch = 0
    best_state = None
    stale_epochs = 0
    history = []
    start = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        pred = model(d["X"], edge_index, edge_weight)
        loss = F.mse_loss(pred[train_mask], d["y"][train_mask])
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            pred_eval = model(d["X"], edge_index, edge_weight)
            train_mse_std = F.mse_loss(pred_eval[train_mask], d["y"][train_mask]).item()
            val_mse_std = F.mse_loss(pred_eval[val_mask], d["y"][val_mask]).item()
            val_rmse = float(np.sqrt(val_mse_std) * d["y_std"])

        history.append({
            "epoch": epoch,
            "train_MSE_standardized": float(loss.item()),
            "train_MSE_C2_eval": train_mse_std * d["y_std"] ** 2,
            "val_RMSE_C": val_rmse,
        })

        if val_rmse < best_val:
            best_val = val_rmse
            best_epoch = epoch
            stale_epochs = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale_epochs += 1

        if stale_epochs >= int(params["patience"]):
            break

    elapsed = time.perf_counter() - start
    pd.DataFrame(history).to_csv(folder / "history.csv", index=False)
    torch.save(best_state, folder / "best_model.pt")

    run = {
        "experiment": experiment, "model": spec["model"], "graph": spec["graph"],
        "init_seed": int(init_seed), "params": params, "feature_names": d["features"],
        "best_epoch": int(best_epoch), "final_epoch": int(epoch),
        "best_val_RMSE_C": float(best_val), "max_epochs": int(max_epochs),
        "stop_reason": "early_stopping" if stale_epochs >= int(params["patience"]) else "epoch_limit",
        "hit_epoch_limit": bool(epoch == max_epochs),
        "parameters": int(sum(p.numel() for p in model.parameters())),
        "seconds": float(elapsed),
    }
    save_json(folder / "run.json", run)

    del model, optimizer, best_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return run

###############################################################
def tune_one(date, group, experiment, split, d):
    folder = OUT / date / group / experiment / f"split_{split}"
    tuning_dir = folder / "tuning"
    tuning_dir.mkdir(parents=True, exist_ok=True)
    params_file = folder / "best_params.json"

    # Skip tuning if already finished
    if params_file.exists():
        return read_json(params_file)["params"]

    model_name = EXPERIMENTS[experiment]["model"]
    sampler_seed = split + 10000 * MODEL_INDEX[model_name]
    study = optuna.create_study(
        study_name="validation_rmse",
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=sampler_seed, n_startup_trials=5),
        pruner=optuna.pruners.NopPruner(),
        storage=f"sqlite:///{folder / 'optuna.db'}",
        load_if_exists=True,
    )

    def objective(trial):
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
            "n_layers": trial.suggest_categorical("n_layers", [1, 2, 3]),
            "hidden_dim": trial.suggest_categorical("hidden_dim", [32, 64, 128]),
            "activation": trial.suggest_categorical("activation", ["relu", "elu", "gelu"]),
            "patience": trial.suggest_categorical("patience", [50, 100, 150]),
        }
        run = fit_one(d, experiment, params, TUNING_SEED, TUNE_MAX_EPOCHS, tuning_dir / f"trial_{trial.number:03d}")
        return run["best_val_RMSE_C"]

    done_trials = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    remaining = N_TRIALS - done_trials
    if remaining > 0:
        study.optimize(objective, n_trials=remaining, n_jobs=1, gc_after_trial=True)

    best_record = {
        "date": date, "feature_group": group, "experiment": experiment,
        "split_seed": int(split), "best_trial_number": int(study.best_trial.number),
        "tuning_val_RMSE_C": float(study.best_value), "params": study.best_trial.params,
        "complete_trials": len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]),
    }
    save_json(params_file, best_record)
    study.trials_dataframe().to_csv(folder / "optuna_trials.csv", index=False)
    return best_record["params"]

## Run hyperparameter tuning and final training

In [ ]:
for date in RUN_DATES:
    print(f"=== Running date: {date} ===")
    for group in GROUPS:
        for experiment in EXPERIMENTS:
            for split in SPLIT_SEEDS:
                d = load_data(date, group, split)
                params = tune_one(date, group, experiment, split, d)
                split_folder = OUT / date / group / experiment / f"split_{split}"

                for init_seed in FINAL_SEEDS:
                    run_folder = split_folder / f"seed_{init_seed}"
                    if (run_folder / "done.json").exists():
                        continue

                    run = fit_one(d, experiment, params, init_seed, FINAL_MAX_EPOCHS, run_folder)
                    run.update({"date": date, "feature_group": group, "split_seed": int(split)})
                    save_json(run_folder / "run.json", run)
                    save_json(run_folder / "done.json", {"completed": True})

                del d
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    # Backup after each date finishes
    shutil.make_archive(str(OUT) + "_backup", "zip", root_dir=OUT.parent, base_dir=OUT.name)
    print(f"Finished and backed up: {date}")

## Evaluate models and summarize results

In [ ]:
def date_is_complete(date):
    for group in GROUPS:
        for exp in EXPERIMENTS:
            for split in SPLIT_SEEDS:
                for seed in FINAL_SEEDS:
                    if not (OUT / date / group / exp / f"split_{split}" / f"seed_{seed}" / "done.json").exists():
                        return False
    return True

completed_dates = [d for d in ALL_DATES if date_is_complete(d)]
print("Completed dates:", completed_dates)

metric_rows, run_rows, best_params_rows = [], [], []

for date in completed_dates:
    for group in GROUPS:
        for experiment, spec in EXPERIMENTS.items():
            edge_index, edge_weight = GRAPHS[spec["graph"]]
            for split in SPLIT_SEEDS:
                d = load_data(date, group, split)
                split_folder = OUT / date / group / experiment / f"split_{split}"
                selected = read_json(split_folder / "best_params.json")
                params = selected["params"]

                best_params_rows.append({
                    "date": date, "feature_group": group, "experiment": experiment,
                    "split_seed": split, "tuning_val_RMSE_C": selected["tuning_val_RMSE_C"],
                    **params,
                })

                for init_seed in FINAL_SEEDS:
                    folder = split_folder / f"seed_{init_seed}"
                    run = read_json(folder / "run.json")

                    model = RegressionModel(spec["model"], d["X"].shape[1], params).to(DEVICE)
                    state = torch.load(folder / "best_model.pt", map_location=DEVICE, weights_only=True)
                    model.load_state_dict(state)
                    model.eval()

                    with torch.no_grad():
                        pred_std = model(d["X"], edge_index, edge_weight).cpu().numpy()
                    preds = pred_std.astype(np.float64) * d["y_std"] + d["y_mean"]

                    # Save per-run metrics
                    cur_metrics = []
                    for subset, mask in d["masks_np"].items():
                        row = {
                            "date": date, "feature_group": group, "experiment": experiment,
                            "model": spec["model"], "graph": spec["graph"],
                            "split_seed": split, "init_seed": init_seed, "subset": subset,
                            **regression_metrics(d["y_raw"][mask], preds[mask]),
                        }
                        metric_rows.append(row)
                        cur_metrics.append(row)
                    pd.DataFrame(cur_metrics).to_csv(folder / "metrics.csv", index=False)

                    # Save per-node predictions
                    subset_names = np.full(N, "", dtype=object)
                    for subset, mask in d["masks_np"].items():
                        subset_names[mask] = subset

                    pred_df = node_order[["local_node_id", "SEZ21_ID"]].copy()
                    pred_df["date"] = date
                    pred_df["subset"] = subset_names
                    pred_df["observed_lst"] = d["y_raw"]
                    pred_df["predicted_lst"] = preds
                    pred_df["error_C"] = preds - d["y_raw"]
                    pred_df.to_csv(folder / "predictions.csv", index=False)

                    run_rows.append({
                        "date": date, "feature_group": group, "experiment": experiment,
                        "split_seed": split, "init_seed": init_seed,
                        "best_epoch": run["best_epoch"], "final_epoch": run["final_epoch"],
                        "best_val_RMSE_C": run["best_val_RMSE_C"], "stop_reason": run["stop_reason"],
                        "hit_epoch_limit": run["hit_epoch_limit"], "seconds": run["seconds"],
                        "parameters": run["parameters"],
                    })

# Aggregate: average over init_seeds first, then mean & std over splits
metrics = pd.DataFrame(metric_rows)
training_runs = pd.DataFrame(run_rows)
best_params_table = pd.DataFrame(best_params_rows)

metric_cols = ["MAE", "RMSE", "R2", "Bias"]
split_keys = ["date", "feature_group", "experiment", "model", "graph", "split_seed", "subset"]
summary_keys = ["date", "feature_group", "experiment", "model", "graph", "subset"]

per_split = metrics.groupby(split_keys, as_index=False)[metric_cols].mean()
summary = per_split.groupby(summary_keys)[metric_cols].agg(["mean", "std"])
summary.columns = [f"{m}_{s}" for m, s in summary.columns]
summary = summary.reset_index()

metrics.to_csv(OUT / "metrics_all_runs.csv", index=False)
per_split.to_csv(OUT / "metrics_per_split.csv", index=False)
summary.to_csv(OUT / "summary.csv", index=False)
training_runs.to_csv(OUT / "training_runs.csv", index=False)
best_params_table.to_csv(OUT / "best_params_all_splits.csv", index=False)

# Paired comparison: public vs all features
paired = per_split.pivot(
    index=["date", "experiment", "model", "graph", "split_seed", "subset"],
    columns="feature_group", values="RMSE"
).reset_index()
paired.columns.name = None
paired["public_minus_all_RMSE_C"] = paired["public"] - paired["all"]
paired.to_csv(OUT / "paired_public_vs_all.csv", index=False)

# Paired comparison: Euclidean vs contiguity graph (GCN & GraphConv)
graph_cmp = per_split[per_split["model"].isin(["GCN", "GraphConv"])].pivot(
    index=["date", "feature_group", "model", "split_seed", "subset"],
    columns="graph", values="RMSE"
).reset_index()
graph_cmp.columns.name = None
graph_cmp["euclidean_minus_contiguity_RMSE_C"] = graph_cmp["euclidean"] - graph_cmp["contiguity"]
graph_cmp.to_csv(OUT / "paired_graph_comparison.csv", index=False)

#####################################
# Show test set summary
test_summary = summary[summary["subset"] == "test"][[
    "date", "feature_group", "experiment",
    "MAE_mean", "MAE_std", "RMSE_mean", "RMSE_std",
    "R2_mean", "R2_std", "Bias_mean", "Bias_std"
]].sort_values(["date", "feature_group", "RMSE_mean"]).round(4)
display(test_summary)

save_json(OUT / "completion_status.json", {
    "completed_dates": completed_dates,
    "remaining_dates": [d for d in ALL_DATES if d not in completed_dates],
    "final_runs_evaluated": len(training_runs),
})
shutil.make_archive(str(OUT) + "_backup", "zip", root_dir=OUT.parent, base_dir=OUT.name)